In [1]:
# Cell 1 – Install dependencies

!pip install datasets huggingface_hub


In [2]:
# Cell 2 – Authenticate with HuggingFace
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret('HF_TOKEN')

if hf_token:
    login(token=hf_token)




In [3]:
# Cell 3 – Reload Raw ChatDoctor and RE-clean with Extended Patterns

from datasets import load_dataset
import re

ds = load_dataset("lavita/ChatDoctor-HealthCareMagic-100k")

def clean_output(text):
    filler_starts = [
        r'^Hello[\w\s,]*Welcome to Chat\s?Doctor[.\s]*',
        r'^Hi[\w\s,]*Welcome to Chat\s?Doctor[.\s]*',
        r'^Thanks for (using|writing to) Chat\s?Doctor[.\s]*',
        r'^Hello and welcome to Chat\s?Doctor[.\s]*',
        r'^and I hope I can help you today\.?[\s]*',
        r'^Thank you for (posting|consulting|writing|using|asking)[\w\s,]*[.\s]+',
        r'^Welcome (to|back)[\w\s,]*[.\s]+',
        r'^Hello[\s,]+', r'^Hi[\s,]+', r'^HI[\s,]+', r'^Hey[\s,]+',
        r'^Dear[\w\s,]+,[\s]*',
    ]

    for pattern in filler_starts:
        text = re.sub(pattern, '', text, flags=re.IGNORECASE).strip()

    filler_ends = [
        r'[,.]?\s*Best wishes,?\s*(Chat\s?Doctor)?\.?$',
        r'[,.]?\s*I hope (this|it) helps?\.?$',
        r'[,.]?\s*Hope (this|it) (helps|answers your question)\.?$',
        r'[,.]?\s*I hope (this|it) (answers?|address(es)?) your (query|question)\.?$',
        r'[,.]?\s*Let me know if (I can assist you further|you have any (further|other) questions)\.?$',
        r'[,.]?\s*(Feel free to|Please) (ask|reach out)[\w\s,]*\.?$',
        r'[,.]?\s*Take care\.?$',
        r'[,.]?\s*Regards,?[\w\s]*\.?$',
        r'[,.]?\s*Wish(ing)? you (good health|a speedy recovery)[\w\s,]*\.?$',
        r'[,.]?\s*Thank you\.?$',
        r'[,.]?\s*Chat\s?Doctor\.?$',
    ]

    # Run end pattern twice: stripping one sign-off can expose another underneath
    for _ in range(2):
        for pattern in filler_ends:
            text = re.sub(pattern, '', text, flags=re.IGNORECASE).strip()

    return text

def is_clean(sample):
    if re.search(r'chatdoctor', sample['input'], re.IGNORECASE):
        return False
    if len(sample['output'].split()) < 30:
        return False
    if len(sample['input'].split()) + len(sample['output'].split()) > 600:
        return False
    return True

def is_clean_after_mapping(sample):
    out = sample['output']
    if re.search(r'chat\s?doctor', out, re.IGNORECASE):
        return False
    if re.match(r'^(and |thank you|hello|hi\b|hey\b|welcome)', out, re.IGNORECASE):
        return False
    # Reject if it still ends with social sign-off
    if re.search(r'(hope this helps|let me know|take care|best wishes|regards)\W*$', out, re.IGNORECASE):
        return False
    return True

chatdoctor = ds['train'].filter(is_clean)
chatdoctor = chatdoctor.map(lambda s: {'output': clean_output(s['output'])})
chatdoctor = chatdoctor.filter(is_clean_after_mapping)

print(f"ChatDoctor clean rows: {len(chatdoctor)}")

README.md:   0%|          | 0.00/542 [00:00<?, ?B/s]

data/train-00000-of-00001-5e7cb295b9cff0(…):   0%|          | 0.00/70.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/112165 [00:00<?, ? examples/s]

Filter:   0%|          | 0/112165 [00:00<?, ? examples/s]

Map:   0%|          | 0/107068 [00:00<?, ? examples/s]

Filter:   0%|          | 0/107068 [00:00<?, ? examples/s]

ChatDoctor clean rows: 45427


In [4]:
# Cell 4 Load and Inspect WikiDoc

wikidoc = load_dataset("medalpaca/medical_meadow_wikidoc")
print(wikidoc)

for i in range(5):
    s = wikidoc['train'][i]
    print(f"INSTRUCTION: {s.get('instruction', 'N/A')}")
    print(f"INPUT: {str(s.get('input', ''))[:200]}")
    print(f"OUTPUT: {str(s.get('output', ''))[:300]}")
    print("-" * 60)

README.md: 0.00B [00:00, ?B/s]

medical_meadow_wikidoc.json:   0%|          | 0.00/10.6M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/10000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input', 'output', 'instruction'],
        num_rows: 10000
    })
})
INSTRUCTION: Answer this question truthfully
INPUT: Can you provide an overview of the lung's squamous cell carcinoma?
OUTPUT: Squamous cell carcinoma of the lung may be classified according to the WHO histological classification system into 4 main types: papillary, clear cell, small cell, and basaloid.
------------------------------------------------------------
INSTRUCTION: Answer this question truthfully
INPUT: What does "Clear: cell" mean?
OUTPUT: Clear cell tumors are part of the surface epithelial-stromal tumor group of Ovarian cancers, accounting for 6% of these neoplastic cases. Clear cell tumors are also associated with the pancreas and salivary glands.
Benign and borderline variants of this neoplasm are rare, and most cases are malignan
------------------------------------------------------------
INSTRUCTION: Answer this question truthfully
INPUT: Can you

In [5]:
# CELL 5 – Clean WikiDoc
def wikidoc_is_clean(sample):
    out = str(sample['output'])
    inp = str(sample['input'])

    if len(inp.split()) + len(out.split()) > 600:
        return False
    return True

wikidoc_clean = wikidoc['train'].filter(wikidoc_is_clean)
print(f"WikiDocclean rows: {len(wikidoc_clean)}")

Filter:   0%|          | 0/10000 [00:00<?, ? examples/s]

WikiDocclean rows: 9668


In [6]:
# CELL 6 – Sample, Combine, Shuffle
from datasets import concatenate_datasets

chatdoctor_sample = chatdoctor.shuffle(seed=42).select(range(8000))
wikidoc_sample = wikidoc_clean.shuffle(seed=42).select(range(4000))

combined = concatenate_datasets([chatdoctor_sample, wikidoc_sample])
combined = combined.shuffle(seed=42)

print(f"Combined: {len(combined)} rows")

Combined: 12000 rows


In [7]:
import re

FILLER_KEYWORDS = [
    'hope i have answered', 'hope this answer', 'hope this helps',
    'hope it helps', 'hope you find', 'hope your concern',
    'let me know if', 'feel free to', 'do not hesitate',
    'happy to help', 'happy to answer', 'wishing you', 'wish you',
    'get well soon', 'take care', 'all the best', 'thanks and',
    'i hope to have been', 'available for further', 'good luck',
    'please use this url', 'direct question to me', 'rate this',
    'welcome for any', 'welcome to any', 'stay healthy', 'be positive',
    'god bless', 'thanks for using', 'contacting us', 'for follow-ups'
]

def clean_text(text):
    # Strip trailing sentences that are filler
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    while sentences:
        if any(kw in sentences[-1].lower() for kw in FILLER_KEYWORDS):
            sentences.pop()
        else:
            break
    text = ' '.join(sentences).strip()
    # Remove truncated signature fragment
    text = re.sub(r'\s*Jay In\s*$', '', text).strip()
    return text

def is_clean_row(sample):
    out = sample['output']
    # Drop wiki markup, talk-page artifacts, URL residue
    if re.search(r'Template:|{{|}}|Category::|#ev:|WikiDoc Sources|http|\(UTC\)|unblock|talk page|sic\)|wikipedia', out, re.IGNORECASE):
        return False
    # Drop rows too short after cleaning
    if len(out.split()) < 30:
        return False
    return True

# Clean text, then filter in one pass
combined = combined.map(lambda s: {'output': clean_text(s['output'])})
combined = combined.filter(is_clean_row)

print(f"After cleaning and filter: {len(combined)}")

Map:   0%|          | 0/12000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/12000 [00:00<?, ? examples/s]

After cleaning and filter: 10565


In [8]:
# Cell 7 – Format With the Same System Prompt
SYSTEM = "If you are a doctor, please answer the medical questions based on the patient's description."

def format_sample(sample):
    return {
        "text": (
            f"<|begin_of_text|>"
            f"<|start_header_id|>system<|end_header_id|>"
            f"{SYSTEM}\n"
            f"<|eot_id|>"
            f"<|start_header_id|>user<|end_header_id|>"
            f"{sample['input']}\n"
            f"<|eot_id|>"
            f"<|start_header_id|>assistant<|end_header_id|>"
            f"{sample['output']}"
            f"<|eot_id|>"
        )
    }

formatted = combined.map(format_sample)
print(formatted[0]['text'])

Map:   0%|          | 0/10565 [00:00<?, ? examples/s]

<|begin_of_text|><|start_header_id|>system<|end_header_id|>If you are a doctor, please answer the medical questions based on the patient's description.
<|eot_id|><|start_header_id|>user<|end_header_id|>i have a slight blood coagulating problem. i had my tonsils and adnoids removed and had a slight bleeding problem when i was 17. they diagnosed me with a very mild enzyme clotting problem that would only affect me if i had a serious surgery. it doesent effect daily life and isnt a serious coagulating problem, very very mild to say the least. well ive heard different storys about how theres waivers and such for this in certain branches. i just wanted to know if the army will take me cause its been a life long dream to serve. and this would be a terrible reason not to get that oppurtunity. i take no medication and this was over 3 years ago. So my main question is it possible for this to get medically waiverd. Thanks
<|eot_id|><|start_header_id|>assistant<|end_header_id|>HiT hanks for your 

In [11]:
# CELL 8 – The Spot Check
import random

random.seed(7)

indices = random.sample(range(len(formatted)), 50)

for i in indices:
    out = combined[i]['output']
    print(f"START: {out[:80]}")
    print(f"END: {out[-80:]}")
    print("-" * 60)
    

START: Morton's neuroma can make walking difficult. Persons with this foot condition ma
END:  an automobile. It may hurt to wear certain types of shoes, such as high- heels.
------------------------------------------------------------
START: The structural chromosomal alterations involving 22q13.1 in osteoid osteoma may 
END: ogen for cells of mesenchymal origin and involved in the transformation process.
------------------------------------------------------------
START: Dear patient Pain and numbness over outer thigh after surgery may be due to 1. I
END: on one at bedtime for 10 days. Please consult expert neurophysician with report.
------------------------------------------------------------
START: Thanks for writing in. Tanning and pigmentation over face are your problems. Mom
END:  a dermatologist for some clinic based procedure like chemical peeling and laser
------------------------------------------------------------
START: I have studied your case. There is possibility of kid

In [12]:
# CELL 9 – Split and Push
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-3B-Instruct")

lengths = [len(tokenizer(s['text'], truncation=False)['input_ids']) for s in formatted]

print(f"Average: {sum(lengths)/len(lengths):.0f} | Max: {max(lengths)} | Over 512: {sum(1 for l in lengths if l > 512)}")

keep = [i for i, l in enumerate(lengths) if l <= 512]
formatted = formatted.select(keep)
print(f"After length filter: {len(formatted)}")

print(formatted[0].keys())

# split and push the clean version
split = formatted.train_test_split(test_size=0.1, seed=42)
split['train'].push_to_hub("nicholas-ugbala-hf/medical-qa-combined-10.5k", split="train")
split['test'].push_to_hub("nicholas-ugbala-hf/medical-qa-combined-10.5k", split="eval")
print(f"Train: {len(split['train'])} | Eval: {len(split['test'])}")


config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Average: 247 | Max: 1402 | Over 512: 310
After length filter: 10255
dict_keys(['instruction', 'input', 'output', 'text'])


...

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Train: 9229 | Eval: 1026
